<a href="https://colab.research.google.com/github/rasia92015-byte/Boltz2_colab/blob/main/notebooks/databases/curated/002_fetch_smiles_from_ChEMBL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Run once if needed:
# !pip install requests openpyxl

from pathlib import Path
import re
import time
import unicodedata

import requests
from openpyxl import load_workbook
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# ============================================================
# USER SETTINGS
# ============================================================

INPUT_FILE = Path("2B_GPCR_Ligand_Master.xlsx")

OUTPUT_FILE = Path(
    "2B_GPCR_Ligand_Master_SMILES.xlsx"
)

LIGAND_COLUMN = "Ligand_Name"

# Delay between unique ligand queries.
REQUEST_DELAY_SECONDS = 0.20

# Number of broad-search results to inspect.
SEARCH_LIMIT = 50


# Optional manual aliases.
# Add entries only when your Excel name differs substantially
# from the name/synonyms used in ChEMBL.
MANUAL_NAME_MAP = {
    # "ADRENALIN": "EPINEPHRINE",
    # "NORADRENALIN": "NOREPINEPHRINE",
}


# Columns that will be written to the output workbook.
OUTPUT_COLUMNS = [
    "SMILES",
    "Compound ChEMBL ID",
    "Standard InChI Key",
    "ChEMBL Preferred Name",
    "ChEMBL Initial Match ID",
    "ChEMBL Parent Resolution",
    "ChEMBL Match Status",
]


# ============================================================
# TERMS INDICATING SALTS, SOLVATES, OR HYDRATES
# ============================================================

FORM_TERMS = {
    # Common acid-addition salts
    "HYDROCHLORIDE",
    "HCL",
    "HYDROBROMIDE",
    "HBR",
    "HYDROIODIDE",
    "SULFATE",
    "SULPHATE",
    "BISULFATE",
    "BISULPHATE",
    "PHOSPHATE",
    "DIPHOSPHATE",
    "NITRATE",
    "ACETATE",
    "CITRATE",
    "TARTRATE",
    "BITARTRATE",
    "MALEATE",
    "FUMARATE",
    "SUCCINATE",
    "OXALATE",
    "LACTATE",
    "GLUCONATE",
    "GLUCURONATE",
    "MESYLATE",
    "METHANESULFONATE",
    "BESYLATE",
    "BENZENESULFONATE",
    "TOSYLATE",
    "ESYLATE",
    "CAMSYLATE",
    "PAMOATE",
    "EMBONATE",
    "SALICYLATE",
    "BENZOATE",
    "MALATE",
    "ASPARTATE",
    "GLUTAMATE",
    "FORMATE",
    "PROPIONATE",
    "BUTYRATE",
    "PERCHLORATE",

    # Counterions
    "SODIUM",
    "POTASSIUM",
    "CALCIUM",
    "MAGNESIUM",
    "LITHIUM",
    "AMMONIUM",
    "CHLORIDE",
    "BROMIDE",
    "IODIDE",

    # Hydrates and solvates
    "HYDRATE",
    "MONOHYDRATE",
    "DIHYDRATE",
    "TRIHYDRATE",
    "TETRAHYDRATE",
    "PENTAHYDRATE",
    "HEXAHYDRATE",
    "HEPTAHYDRATE",
    "HEMIHYDRATE",
    "SESQUIHYDRATE",
    "SOLVATE",
    "ETHANOLATE",
    "METHANOLATE",
    "ACETONATE",

    # General form words
    "SALT",
}


# ============================================================
# REQUEST SESSION WITH RETRIES
# ============================================================

retry_strategy = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
)

session = requests.Session()

session.mount(
    "https://",
    HTTPAdapter(max_retries=retry_strategy),
)

session.headers.update(
    {
        "User-Agent": "GPCR-Ligand-Parent-ChEMBL-Annotation/1.0",
        "Accept": "application/json",
    }
)

API_BASE = "https://www.ebi.ac.uk/chembl/api/data"


# Cache API records so the same ChEMBL ID is fetched only once.
molecule_cache = {}


# ============================================================
# TEXT NORMALIZATION
# ============================================================

def normalize_name(value):
    """
    Normalize a chemical name for comparison.

    Example:
        'Prazosin hydrochloride' -> 'PRAZOSIN HYDROCHLORIDE'
    """

    if value is None:
        return ""

    text = unicodedata.normalize("NFKD", str(value))
    text = text.encode("ascii", "ignore").decode("ascii")
    text = text.upper()

    # Replace punctuation with spaces.
    text = re.sub(r"[^A-Z0-9]+", " ", text)

    return " ".join(text.split())


def contains_form_term(value):
    """
    Return True when a name contains a common salt,
    hydrate, solvate, or counterion term.
    """

    normalized = normalize_name(value)
    words = set(normalized.split())

    return bool(words.intersection(FORM_TERMS))


def get_synonym_names(molecule):
    """Return normalized synonyms from a ChEMBL molecule record."""

    synonyms = set()

    for item in molecule.get("molecule_synonyms") or []:
        if not isinstance(item, dict):
            continue

        synonym = (
            item.get("molecule_synonym")
            or item.get("synonyms")
            or item.get("synonym")
        )

        if synonym:
            synonyms.add(normalize_name(synonym))

    return synonyms


# ============================================================
# ChEMBL API FUNCTIONS
# ============================================================

def api_get(url, params=None):
    """Make a GET request and return parsed JSON."""

    response = session.get(
        url,
        params=params,
        timeout=45,
    )

    response.raise_for_status()
    return response.json()


def get_molecule_by_id(chembl_id):
    """Retrieve one full ChEMBL molecule record."""

    if not chembl_id:
        return None

    chembl_id = str(chembl_id).strip().upper()

    if chembl_id in molecule_cache:
        return molecule_cache[chembl_id]

    url = f"{API_BASE}/molecule/{chembl_id}.json"

    try:
        record = api_get(url)
    except requests.RequestException:
        return None

    molecule_cache[chembl_id] = record
    return record


def exact_preferred_name_search(name):
    """
    Search by exact case-insensitive ChEMBL preferred name.
    """

    url = f"{API_BASE}/molecule.json"

    data = api_get(
        url,
        params={
            "pref_name__iexact": name,
            "limit": SEARCH_LIMIT,
        },
    )

    return data.get("molecules", [])


def broad_molecule_search(name):
    """Search ChEMBL molecule names and synonyms."""

    url = f"{API_BASE}/molecule/search.json"

    data = api_get(
        url,
        params={
            "q": name,
            "limit": SEARCH_LIMIT,
        },
    )

    return data.get("molecules", [])


# ============================================================
# PARENT COMPOUND RESOLUTION
# ============================================================

def resolve_parent_molecule(molecule):
    """
    Follow molecule_hierarchy.parent_chembl_id until the final
    parent compound is reached.

    Returns:
        parent_record
        resolution_path
    """

    if not molecule:
        return None, []

    current = molecule
    path = []

    # Protection against unexpected hierarchy loops.
    for _ in range(10):
        current_id = current.get("molecule_chembl_id")

        if current_id:
            path.append(current_id)

        hierarchy = current.get("molecule_hierarchy") or {}
        parent_id = hierarchy.get("parent_chembl_id")

        # No parent information: treat current record as final.
        if not parent_id:
            break

        parent_id = str(parent_id).upper()

        # Parent points to itself: current record is the parent.
        if parent_id == current_id:
            break

        # Prevent loops.
        if parent_id in path:
            break

        parent_record = get_molecule_by_id(parent_id)

        if not parent_record:
            break

        current = parent_record

    return current, path


# ============================================================
# CANDIDATE SCORING
# ============================================================

def text_match_score(query_name, molecule):
    """
    Score how closely a ChEMBL parent compound matches the
    requested Ligand_Name.
    """

    query = normalize_name(query_name)
    preferred_name = normalize_name(molecule.get("pref_name"))
    synonyms = get_synonym_names(molecule)

    score = 0
    match_type = "weak search match"

    # Strongest: exact preferred name
    if preferred_name == query:
        score += 1000
        match_type = "exact preferred name"

    # Exact synonym
    elif query in synonyms:
        score += 900
        match_type = "exact synonym"

    # Query appears fully in preferred name
    elif query and query in preferred_name:
        score += 500
        match_type = "preferred-name partial match"

    # Preferred name appears fully in the query
    elif preferred_name and preferred_name in query:
        score += 450
        match_type = "preferred-name reverse partial match"

    # Query appears in a synonym
    elif any(query and query in synonym for synonym in synonyms):
        score += 400
        match_type = "synonym partial match"

    # Token comparison
    else:
        query_tokens = set(query.split())
        preferred_tokens = set(preferred_name.split())

        if query_tokens and query_tokens.issubset(preferred_tokens):
            score += 300
            match_type = "preferred-name token match"

        elif any(
            query_tokens
            and query_tokens.issubset(set(synonym.split()))
            for synonym in synonyms
        ):
            score += 250
            match_type = "synonym token match"

    return score, match_type


def score_parent_candidate(query_name, parent_record):
    """
    Prefer the original parent compound rather than salts,
    hydrates, or multi-component structures.
    """

    text_score, match_type = text_match_score(
        query_name,
        parent_record,
    )

    score = text_score

    preferred_name = parent_record.get("pref_name") or ""
    structures = parent_record.get("molecule_structures") or {}
    smiles = structures.get("canonical_smiles") or ""

    # Prefer records that actually have a structure.
    if smiles:
        score += 100
    else:
        score -= 1000

    # Strongly prefer names without salt/hydrate words.
    if contains_form_term(preferred_name):
        score -= 500
    else:
        score += 100

    # A dot normally means multiple disconnected components,
    # such as active molecule + counterion or solvent.
    if "." in smiles:
        score -= 250
    else:
        score += 75

    # Small bonus for small-molecule records.
    molecule_type = normalize_name(
        parent_record.get("molecule_type")
    )

    if molecule_type == "SMALL MOLECULE":
        score += 25

    return score, match_type


# ============================================================
# LOOK UP ONE LIGAND
# ============================================================

def empty_result(status):
    return {
        "smiles": None,
        "chembl_id": None,
        "inchi_key": None,
        "preferred_name": None,
        "initial_match_id": None,
        "parent_resolution": None,
        "status": status,
    }


def lookup_parent_compound(original_name):
    """
    Find the ChEMBL parent compound for one Ligand_Name.
    """

    if original_name is None:
        return empty_result("Blank Ligand_Name")

    original_name = str(original_name).strip()

    if not original_name:
        return empty_result("Blank Ligand_Name")

    normalized_original = normalize_name(original_name)

    search_name = MANUAL_NAME_MAP.get(
        normalized_original,
        original_name,
    )

    try:
        # First, use exact preferred-name matching.
        candidates = exact_preferred_name_search(search_name)
        search_method = "exact preferred-name search"

        # Use broad search only when exact preferred name fails.
        if not candidates:
            candidates = broad_molecule_search(search_name)
            search_method = "broad molecule search"

    except requests.RequestException as error:
        return empty_result(f"ChEMBL request error: {error}")

    if not candidates:
        return empty_result("No ChEMBL search result")

    resolved_candidates = {}

    for search_rank, candidate in enumerate(candidates, start=1):
        initial_id = candidate.get("molecule_chembl_id")

        if not initial_id:
            continue

        # Retrieve the complete molecule record.
        full_candidate = get_molecule_by_id(initial_id)

        if not full_candidate:
            full_candidate = candidate

        # Convert salt/hydrate/form result to its parent compound.
        parent_record, resolution_path = resolve_parent_molecule(
            full_candidate
        )

        if not parent_record:
            continue

        parent_id = parent_record.get("molecule_chembl_id")

        if not parent_id:
            continue

        score, match_type = score_parent_candidate(
            search_name,
            parent_record,
        )

        candidate_info = {
            "score": score,
            "match_type": match_type,
            "search_rank": search_rank,
            "search_method": search_method,
            "initial_id": initial_id,
            "parent_record": parent_record,
            "resolution_path": resolution_path,
        }

        # Different salts may resolve to the same parent.
        # Keep the best-scoring route to each parent.
        previous = resolved_candidates.get(parent_id)

        if previous is None:
            resolved_candidates[parent_id] = candidate_info

        elif candidate_info["score"] > previous["score"]:
            resolved_candidates[parent_id] = candidate_info

        elif (
            candidate_info["score"] == previous["score"]
            and candidate_info["search_rank"]
            < previous["search_rank"]
        ):
            resolved_candidates[parent_id] = candidate_info

    if not resolved_candidates:
        return empty_result(
            "Search results found, but parent could not be resolved"
        )

    # Sort by:
    # 1. highest compound/name score
    # 2. earliest ChEMBL search rank
    ranked = sorted(
        resolved_candidates.values(),
        key=lambda item: (
            -item["score"],
            item["search_rank"],
        ),
    )

    best = ranked[0]
    parent = best["parent_record"]

    structures = parent.get("molecule_structures") or {}

    smiles = structures.get("canonical_smiles")
    inchi_key = structures.get("standard_inchi_key")
    parent_id = parent.get("molecule_chembl_id")
    preferred_name = parent.get("pref_name")

    if not smiles:
        return empty_result(
            "Best parent compound has no canonical SMILES"
        )

    resolution_path = best["resolution_path"]

    if (
        best["initial_id"]
        and parent_id
        and best["initial_id"] != parent_id
    ):
        resolution_description = (
            f"{best['initial_id']} -> {parent_id}"
        )
        resolution_status = "resolved salt/form to parent"
    else:
        resolution_description = parent_id
        resolution_status = "selected parent directly"

    status = (
        f"Found; {resolution_status}; "
        f"{best['match_type']}; "
        f"{best['search_method']}; "
        f"score={best['score']}"
    )

    return {
        "smiles": smiles,
        "chembl_id": parent_id,
        "inchi_key": inchi_key,
        "preferred_name": preferred_name,
        "initial_match_id": best["initial_id"],
        "parent_resolution": resolution_description,
        "status": status,
    }


# ============================================================
# EXCEL FUNCTIONS
# ============================================================

def find_ligand_column(workbook, required_header):
    """
    Search the first 20 rows of every worksheet for Ligand_Name.
    """

    required = normalize_name(required_header)

    for worksheet in workbook.worksheets:
        rows_to_check = min(worksheet.max_row, 20)

        for row_number in range(1, rows_to_check + 1):
            for cell in worksheet[row_number]:
                if normalize_name(cell.value) == required:
                    return worksheet, row_number, cell.column

    raise ValueError(
        f'Could not find the "{required_header}" column '
        "in the first 20 rows of any worksheet."
    )


def create_or_find_output_columns(worksheet, header_row):
    """Reuse existing columns or add new columns."""

    header_map = {}

    for cell in worksheet[header_row]:
        if cell.value is not None:
            header_map[normalize_name(cell.value)] = cell.column

    next_column = worksheet.max_column + 1
    output_map = {}

    for column_name in OUTPUT_COLUMNS:
        normalized = normalize_name(column_name)

        if normalized in header_map:
            output_map[column_name] = header_map[normalized]
        else:
            worksheet.cell(
                row=header_row,
                column=next_column,
                value=column_name,
            )

            output_map[column_name] = next_column
            next_column += 1

    return output_map


# ============================================================
# PROCESS THE WORKBOOK
# ============================================================

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE.resolve()}"
    )

workbook = load_workbook(INPUT_FILE)

worksheet, header_row, ligand_column = find_ligand_column(
    workbook,
    LIGAND_COLUMN,
)

output_map = create_or_find_output_columns(
    worksheet,
    header_row,
)

print(f"Worksheet: {worksheet.title}")
print(f"Header row: {header_row}")
print(f"Ligand column: {ligand_column}")


# Cache repeated ligand names.
ligand_cache = {}

number_of_data_rows = worksheet.max_row - header_row

for position, row_number in enumerate(
    range(header_row + 1, worksheet.max_row + 1),
    start=1,
):
    ligand_value = worksheet.cell(
        row=row_number,
        column=ligand_column,
    ).value

    ligand_name = (
        str(ligand_value).strip()
        if ligand_value is not None
        else ""
    )

    cache_key = normalize_name(ligand_name)

    if cache_key not in ligand_cache:
        print(
            f"[{position}/{number_of_data_rows}] "
            f"Searching: {ligand_name}"
        )

        ligand_cache[cache_key] = lookup_parent_compound(
            ligand_name
        )

        time.sleep(REQUEST_DELAY_SECONDS)

    result = ligand_cache[cache_key]

    worksheet.cell(
        row=row_number,
        column=output_map["SMILES"],
        value=result["smiles"],
    )

    worksheet.cell(
        row=row_number,
        column=output_map["Compound ChEMBL ID"],
        value=result["chembl_id"],
    )

    worksheet.cell(
        row=row_number,
        column=output_map["Standard InChI Key"],
        value=result["inchi_key"],
    )

    worksheet.cell(
        row=row_number,
        column=output_map["ChEMBL Preferred Name"],
        value=result["preferred_name"],
    )

    worksheet.cell(
        row=row_number,
        column=output_map["ChEMBL Initial Match ID"],
        value=result["initial_match_id"],
    )

    worksheet.cell(
        row=row_number,
        column=output_map["ChEMBL Parent Resolution"],
        value=result["parent_resolution"],
    )

    worksheet.cell(
        row=row_number,
        column=output_map["ChEMBL Match Status"],
        value=result["status"],
    )

    print(
        f"    -> {result['preferred_name'] or 'Not found'}"
        f" | {result['chembl_id'] or ''}"
        f" | {result['status']}"
    )

    # Periodic checkpoint.
    if position % 25 == 0:
        workbook.save(OUTPUT_FILE)
        print(f"    Checkpoint saved: {OUTPUT_FILE}")


# ============================================================
# BASIC COLUMN WIDTHS
# ============================================================

widths = {
    "SMILES": 60,
    "Compound ChEMBL ID": 22,
    "Standard InChI Key": 32,
    "ChEMBL Preferred Name": 30,
    "ChEMBL Initial Match ID": 24,
    "ChEMBL Parent Resolution": 32,
    "ChEMBL Match Status": 65,
}

for column_name, width in widths.items():
    column_number = output_map[column_name]

    column_letter = worksheet.cell(
        row=header_row,
        column=column_number,
    ).column_letter

    worksheet.column_dimensions[column_letter].width = width


# ============================================================
# SAVE
# ============================================================

workbook.save(OUTPUT_FILE)

print("\nFinished.")
print(f"Output file: {OUTPUT_FILE.resolve()}")
print(f"Unique ligand names queried: {len(ligand_cache)}")

Worksheet: Sheet1
Header row: 1
Ligand column: 5
[1/161] Searching: epinephrine
    -> EPINEPHRINE | CHEMBL679 | Found; selected parent directly; exact preferred name; exact preferred-name search; score=1300
[2/161] Searching: prazosin
    -> PRAZOSIN | CHEMBL2 | Found; selected parent directly; exact preferred name; exact preferred-name search; score=1300
[3/161] Searching: phenylephrine
    -> PHENYLEPHRINE | CHEMBL1215 | Found; selected parent directly; exact preferred name; exact preferred-name search; score=1300
[4/161] Searching: matched_decoy_to_generate
    -> Not found |  | No ChEMBL search result
    -> EPINEPHRINE | CHEMBL679 | Found; selected parent directly; exact preferred name; exact preferred-name search; score=1300
    -> PRAZOSIN | CHEMBL2 | Found; selected parent directly; exact preferred name; exact preferred-name search; score=1300
    -> PHENYLEPHRINE | CHEMBL1215 | Found; selected parent directly; exact preferred name; exact preferred-name search; score=1300
    